# Importing Libraries and dataset from kaggle

In [ ]:
!pip install -q kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

In [ ]:
import os

print(os.listdir(path))      # see folders/files inside

# The 'train' and 'test' directories are directly under 'chest_xray' subfolder
base_data_dir = os.path.join(path, "chest_xray")
train_dir = os.path.join(base_data_dir, "train")
test_dir = os.path.join(base_data_dir, "test")

print("Train directory:", train_dir, "Content:", os.listdir(train_dir)[:10])
print("Test directory:", test_dir, "Content:", os.listdir(test_dir)[:10])

# Data Preprocessing

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os # Import os for path manipulation

# Ensure 'path' is available from previous steps, e.g., '/kaggle/input/chest-xray-pneumonia'
# Explicitly define train_dir and test_dir using the known correct structure
base_data_dir = os.path.join(path, "chest_xray")
train_dir = os.path.join(base_data_dir, "train")
test_dir = os.path.join(base_data_dir, "test")

img_size = (224, 224)
batch_size = 32

# 2) Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range = 0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=False
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True,
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False,
)

In [ ]:
from sklearn.utils import class_weight
import numpy as np

# Get class labels for all training samples
y_train = train_gen.classes

# Calculate class weights
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to a dictionary mapping class index to weight
class_weights_dict = dict(enumerate(class_weights))

print("Class weights:", class_weights_dict)

# Load MobileNetV2 Base and add the custom head

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization


NUM_CLASSES = train_gen.num_classes

base_model = MobileNetV2( input_shape=(224,224) + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = layers.Input(shape=(224,224) + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x) # This replaces the Flatten layer after feature extraction
x = layers.Dense(128, activation='relu')(x) # Adjusted to 128 units as in your custom model
x = BatchNormalization()(x)
x = layers.Dropout(0.2)(x) # Adjusted to 0.2 dropout as in your custom model
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x) # Uses NUM_CLASSES (which is 2) and softmax for categorical output

Chest_tl_model = models.Model(inputs, outputs)

In [ ]:
print(train_gen.class_indices)

# Compiling

In [ ]:
import tensorflow as tf
Chest_tl_model.compile( optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# Training

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

callbacks = [ReduceLROnPlateau(monitor='val_loss',factor=0.3,patience=2,min_lr=1e-6,verbose=1)]

In [ ]:
history_Chest_tl = Chest_tl_model.fit(train_gen,validation_data=test_gen,epochs=12, callbacks=callbacks)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

base_model.trainable = True
for layer in base_model.layers[:-2]:
    layer.trainable = False

callbacks = [
    ReduceLROnPlateau(monitor='val_loss',factor=0.3,patience=2,min_lr=1e-6,verbose=1),
    EarlyStopping(monitor='val_loss',patience=6,restore_best_weights=True,verbose=1)
]

Chest_tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history = Chest_tl_model.fit( train_gen, validation_data=test_gen, epochs=7, callbacks=callbacks)

# Accuracy

In [ ]:
loss, accuracy = Chest_tl_model.evaluate(test_gen)
print(f"Test loss: {loss}")
print(f"Test accuracy: {accuracy}")

# Confusion Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

test_gen.reset()
test_gen.shuffle = False

predictions = Chest_tl_model.predict(test_gen)
y_pred = np.argmax(predictions, axis=1)

y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Chest X-Ray Classification')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# 6. Detailed Performance Report
print(classification_report(y_true, y_pred, target_names=class_names))

# Single Prediction

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

image_path = 'X ray.jpg'
img_width, img_height = 224, 224

test_image = image.load_img(image_path, target_size=(img_width, img_height))
test_image = image.img_to_array(test_image)

test_image = test_image / 255.0

test_image = np.expand_dims(test_image, axis=0)

result = Chest_tl_model.predict(test_image)

class_indices = train_gen.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

predicted_index = np.argmax(result[0])
prediction = idx_to_class[predicted_index]

print(f"Predicted Type: {prediction}")
print(f"Confidence Scores: {result[0]}")

# Save the model

In [ ]:
Chest_tl_model.save("Chest_model.h5")